In [ ]:
import sys
sys.path.insert(0, "../src")
from TownScapeENV import *
import streamlit as st
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
import base64
from PIL import ImageDraw, ImageFont

In [ ]:
np.zeros(shape = env.observation_shape[1:]).astype(int)
# "0":"ground","1":"obstacle","2":"road","3":"home","4":"target

In [ ]:
shape = (12,17)
env = townScape(shape)
map = np.array([[0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 2, 0, 0, 1, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 2, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 1, 0],
                [0, 3, 2, 2, 2, 0, 0, 1, 0, 2, 0, 3, 0, 4, 0, 4, 0],
                [0, 0, 0, 0, 2, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 1],
                [0, 0, 0, 0, 2, 0, 0, 4, 0, 0, 0, 2, 0, 0, 2, 0, 0],
                [0, 0, 0, 4, 2, 0, 0, 2, 0, 0, 0, 2, 4, 0, 2, 1, 0],
                [0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 0],
                [4, 2, 0, 0, 2, 0, 3, 0, 3, 2, 0, 0, 0, 2, 3, 0, 0],
                [0, 0, 0, 3, 2, 0, 0, 0, 0, 2, 4, 0, 0, 2, 0, 0, 0],
                [0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 3, 0],
                [0, 0, 0, 0, 4, 0, 0, 0, 3, 3, 0, 0, 0, 3, 0, 0, 0]])

obstacles = (map == 1).astype(int)
roads = (map == 2).astype(int)
homes = (map == 3).astype(int)
targets = (map == 4).astype(int)

In [ ]:
req = {"home": np.sum(homes), 'target': np.sum(targets), "map": obstacles}
env.reset(requirements=req)

for mat in [homes, targets]:
    _ = np.where(mat == 1)
    acts = list(zip(_[0], _[1]))
    for act in acts:
        env.step(env.act_encoder(act))
# display_img(env.render())
_ = np.where(roads == 1)
roads = list(zip(_[0], _[1]))
for road in roads:
    env.addRoad(*road)
display_img(env.render())

In [ ]:
map2 = np.array([[0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 2, 0, 0, 1, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 2, 1, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 1, 0],
                [0, 3, 2, 5, 2, 0, 0, 1, 0, 2, 0, 3, 0, 4, 0, 4, 0],
                [0, 0, 0, 0, 2, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 1],
                [0, 0, 0, 0, 2, 0, 0, 4, 0, 0, 0, 5, 0, 0, 2, 0, 0],
                [0, 0, 0, 4, 2, 0, 0, 2, 0, 0, 0, 2, 4, 0, 5, 1, 0],
                [0, 2, 2, 2, 2, 5, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 0],
                [4, 5, 0, 0, 2, 0, 3, 0, 3, 2, 0, 0, 0, 2, 3, 0, 0],
                [0, 0, 0, 3, 2, 0, 0, 0, 0, 2, 4, 0, 0, 2, 0, 0, 0],
                [0, 0, 0, 0, 2, 2, 2, 2, 5, 2, 2, 2, 1, 2, 2, 3, 0],
                [0, 0, 0, 0, 4, 0, 0, 0, 3, 3, 0, 0, 0, 3, 0, 0, 0]])




_ = np.where((map2 == 5) | (map2 == 6))
cars = list(zip(_[0], _[1]))
_ = np.where((map2 == 6))
cars2 = list(zip(_[0], _[1]))
cars += cars2
cars = [Car(pos, (0,0)) for pos in cars]
env.car_list = cars

env.car_space = [[list() for j in range(env.observation_shape[2])] for i in range(env.observation_shape[1])]
for car in cars:
    pos = car.get_pos()
    env.car_space[pos[0]][pos[1]].append(car)
display_img(env.render())

In [ ]:
im2 = Image.fromarray(env.render())
bio = BytesIO()
im2.save('art.png', format='png')